# Lecture 03: Supervised Learning Deep Dive

Runnable demos matching the lecture slides:
1. Linear Regression (house price)
2. Noisy data + multiple candidate lines
3. Residuals and MSE
4. Normal Equation in NumPy
5. Gradient Descent from scratch
6. Learning rate comparison
7. Feature scaling
8. sklearn vs PyTorch: Linear Regression
9. Logistic Regression + sigmoid
10. Logistic Regression in sklearn
11. Logistic Regression in PyTorch
12. Cross-entropy loss
13. Polynomial features

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 13

## 1. Linear Regression: The House Price Example

In [ ]:
# Data from the slides
sizes = np.array([1000, 1500, 2000, 2500])
prices = np.array([40, 60, 80, 100])

plt.scatter(sizes, prices, s=120, color='black', zorder=5)
x_line = np.linspace(800, 2700, 100)
plt.plot(x_line, 0.04 * x_line, color='#2196F3', linewidth=2, label='y = 0.04x')
plt.xlabel('Size (sqft)'); plt.ylabel('Price (lakhs)')
plt.title('Perfect Linear Relationship'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

print("Pattern: every 500 sqft adds 20 lakhs")
print(f"Prediction for 1750 sqft: {0.04 * 1750} lakhs")

## 2. Noisy Data: Which Line is Best?

In [ ]:
# Now with realistic noisy data
prices_noisy = np.array([42, 58, 83, 97])

# Three candidate lines
lines = [
    ("Line A: y = 0.04x",      lambda x: 0.04 * x,       '#2196F3'),
    ("Line B: y = 0.038x + 3", lambda x: 0.038 * x + 3,  '#FF9800'),
    ("Line C: y = 0.035x + 10",lambda x: 0.035 * x + 10, '#E91E63'),
]

plt.scatter(sizes, prices_noisy, s=120, color='black', zorder=5, label='Actual data')
for label, fn, color in lines:
    plt.plot(x_line, fn(x_line), color=color, linewidth=2.5, label=label)
    sse = sum((yi - fn(xi))**2 for xi, yi in zip(sizes, prices_noisy))
    print(f"{label:30s}  SSE = {sse:.0f}")

plt.xlabel('Size (sqft)'); plt.ylabel('Price (lakhs)')
plt.title('Which Line Fits Best?'); plt.legend(); plt.grid(alpha=0.3)
plt.show()
print("\nLine B has the lowest SSE -- it's the best fit!")

## 3. Residuals and MSE

In [ ]:
# Using Line A (y = 0.04x) from the slides
predicted = 0.04 * sizes
residuals = prices_noisy - predicted

print(f"{'Size':>6} {'Actual':>8} {'Predicted':>10} {'Residual':>10} {'Residual^2':>12}")
print("-" * 50)
for s, a, p, r in zip(sizes, prices_noisy, predicted, residuals):
    print(f"{s:>6} {a:>8} {p:>10.0f} {r:>+10.0f} {r**2:>12.0f}")

sse = np.sum(residuals**2)
mse = np.mean(residuals**2)
print(f"\nSSE = {sse:.0f}")
print(f"MSE = SSE / n = {sse:.0f} / {len(sizes)} = {mse:.1f}")

## 4. Normal Equation: Solving Directly

In [ ]:
# The calculus trick: take derivative, set to 0, solve
# theta_hat = (X^T X)^{-1} X^T y

X = np.column_stack([np.ones(len(sizes)), sizes])  # add column of 1s for bias
y = prices_noisy

print("X (augmented):")
print(X)

theta_hat = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"\nNormal equation solution:")
print(f"  bias (theta_0)   = {theta_hat[0]:.2f}")
print(f"  weight (theta_1) = {theta_hat[1]:.4f}")
print(f"\nEquation: y = {theta_hat[1]:.4f}x + {theta_hat[0]:.2f}")
print(f"This is Line B from our plot!")

In [ ]:
# Verify: sklearn gives the same answer
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(sizes.reshape(-1, 1), prices_noisy)
print(f"sklearn: weight = {model.coef_[0]:.4f}, bias = {model.intercept_:.2f}")
print(f"Normal:  weight = {theta_hat[1]:.4f}, bias = {theta_hat[0]:.2f}")
print("Same answer!")

## 5. Gradient Descent from Scratch

In [ ]:
def gradient_descent(X, y, lr=0.01, epochs=1000):
    theta = np.zeros(X.shape[1])  # start with zeros
    history = []

    for epoch in range(epochs):
        y_pred = X @ theta                          # predictions
        error = y - y_pred                          # residuals
        gradient = (-2 / len(y)) * (X.T @ error)   # MSE gradient
        theta = theta - lr * gradient               # update!

        loss = np.mean(error**2)
        history.append(loss)

    return theta, history

# Normalize X for stable gradient descent
X_norm = np.column_stack([np.ones(len(sizes)), sizes / 1000.0])
y_gd = prices_noisy

theta_gd, losses = gradient_descent(X_norm, y_gd, lr=0.1, epochs=200)

print(f"Gradient descent solution (after 200 steps):")
print(f"  theta = [{theta_gd[0]:.2f}, {theta_gd[1]:.2f}]")
print(f"  (weight in original scale = {theta_gd[1]/1000:.4f})")
print(f"  Final MSE = {losses[-1]:.2f}")

In [ ]:
# Plot the loss curve
plt.plot(losses, linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Gradient Descent: Loss Decreasing Over Time')
plt.grid(alpha=0.3)
plt.show()
print("Loss drops quickly at first, then converges.")

## 6. Learning Rate Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, lr, title in zip(axes, [0.001, 0.1, 1.5],
                          ['Too small (0.001)', 'Just right (0.1)', 'Too large (1.5)']):
    _, hist = gradient_descent(X_norm, y_gd, lr=lr, epochs=200)
    ax.plot(hist, linewidth=2)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
    ax.set_ylim(0, max(hist[0] * 1.2, 100))
    ax.grid(alpha=0.3)

plt.suptitle('Effect of Learning Rate', fontsize=15, y=1.02)
plt.tight_layout(); plt.show()
print("Too small: barely moves | Just right: converges fast | Too large: explodes!")

## 7. Feature Scaling: Why It Matters

In [ ]:
from sklearn.preprocessing import StandardScaler

# Different scales
data = np.array([
    [1000, 3],   # size (sqft), bedrooms
    [1500, 2],
    [2000, 4],
    [2500, 3],
])

print("Before scaling:")
print(f"  Size range:     {data[:,0].min()} - {data[:,0].max()}")
print(f"  Bedrooms range: {data[:,1].min()} - {data[:,1].max()}")

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

print(f"\nAfter StandardScaler:")
print(f"  Size:     mean={data_scaled[:,0].mean():.1f}, std={data_scaled[:,0].std():.1f}")
print(f"  Bedrooms: mean={data_scaled[:,1].mean():.1f}, std={data_scaled[:,1].std():.1f}")
print("\nBoth features now speak the same language!")

In [ ]:
# IMPORTANT: fit on train, transform both
from sklearn.model_selection import train_test_split

X_train, X_test = data[:3], data[3:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit AND transform
X_test_scaled  = scaler.transform(X_test)         # only transform!

print("Correct: fit scaler on training data only")
print(f"Train scaled mean: {X_train_scaled.mean(axis=0)}")
print(f"Test scaled mean:  {X_test_scaled.mean(axis=0)}  (not zero -- that's correct!)")

## 8. sklearn vs PyTorch: Linear Regression

In [ ]:
# sklearn: 3 lines
from sklearn.linear_model import LinearRegression

X_sk = sizes.reshape(-1, 1)
model_sk = LinearRegression().fit(X_sk, prices_noisy)

print("=== sklearn ===")
print(f"weight = {model_sk.coef_[0]:.4f}, bias = {model_sk.intercept_:.2f}")
print(f"Prediction for 1750: {model_sk.predict([[1750]])[0]:.1f} lakhs")

In [ ]:
# PyTorch: the 5-step training loop
import torch
import torch.nn as nn

X_pt = torch.tensor(sizes / 1000.0, dtype=torch.float32).reshape(-1, 1)
y_pt = torch.tensor(prices_noisy, dtype=torch.float32).reshape(-1, 1)

model_pt = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model_pt.parameters(), lr=0.1)

for epoch in range(500):
    y_pred = model_pt(X_pt)              # 1. Forward pass
    loss = criterion(y_pred, y_pt)        # 2. Compute loss
    optimizer.zero_grad()                 # 3. Clear gradients
    loss.backward()                       # 4. Compute gradients
    optimizer.step()                      # 5. Update weights

w = model_pt.weight.item() / 1000  # rescale
b = model_pt.bias.item()
print("\n=== PyTorch ===")
print(f"weight = {w:.4f}, bias = {b:.2f}")
pred = model_pt(torch.tensor([[1.75]])).item()
print(f"Prediction for 1750: {pred:.1f} lakhs")
print(f"\nSame answer as sklearn!")

## 9. Logistic Regression: The Sigmoid

In [ ]:
# The sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-6, 6, 200)
plt.plot(z, sigmoid(z), linewidth=3, color='#1e3a5f')
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('z (linear score)'); plt.ylabel('sigma(z)')
plt.title('Sigmoid: Squashes any number to (0, 1)')
plt.grid(alpha=0.3); plt.show()

# Verify the table from slides
print(f"{'z':>5}  {'sigma(z)':>10}  Meaning")
print("-" * 35)
for val in [-5, -2, 0, 2, 5]:
    s = sigmoid(val)
    print(f"{val:>5}  {s:>10.4f}  {'~0%' if s < 0.05 else '50/50' if abs(s-0.5)<0.01 else '~100%' if s > 0.95 else f'{s:.0%}'}")

In [ ]:
# Spam filter example from slides
# Learned weights: w1=0.5 (exclamations), w2=2.0 (has FREE), b=-1.0
w1, w2, b = 0.5, 2.0, -1.0

# Email: 5 exclamation marks, has "FREE"
exclamations, has_free = 5, 1
z = w1 * exclamations + w2 * has_free + b
prob = sigmoid(z)

print(f"Linear score: z = {w1}*{exclamations} + {w2}*{has_free} + ({b}) = {z}")
print(f"P(spam) = sigmoid({z}) = {prob:.4f}")
print(f"Decision: {'SPAM' if prob > 0.5 else 'NOT SPAM'} ({prob:.0%} confidence)")

## 10. Logistic Regression in sklearn

In [ ]:
from sklearn.linear_model import LogisticRegression

# Spam filter data from slides
X_spam = np.array([[5, 1], [0, 0], [3, 1], [1, 0]])  # [exclamations, has_FREE]
y_spam = np.array([1, 0, 1, 0])                       # 1=spam, 0=not spam

model_lr = LogisticRegression()
model_lr.fit(X_spam, y_spam)

# Predict class
print(f"Prediction for [4, 1]: {model_lr.predict([[4, 1]])[0]}  (1 = spam)")

# Predict probabilities
probs = model_lr.predict_proba([[4, 1]])
print(f"P(not spam) = {probs[0][0]:.3f}, P(spam) = {probs[0][1]:.3f}")

print(f"\nLearned weights: {model_lr.coef_[0]}")
print(f"Learned bias:    {model_lr.intercept_[0]:.3f}")

## 11. Logistic Regression in PyTorch

In [ ]:
# Same spam data as PyTorch tensors
X_spam_pt = torch.tensor([[5, 1], [0, 0], [3, 1], [1, 0]], dtype=torch.float32)
y_spam_pt = torch.tensor([[1], [0], [1], [0]], dtype=torch.float32)

# Model: Linear + Sigmoid (exactly as in the slides)
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.linear(x))

model_lr_pt = LogisticRegressionModel(input_dim=2)

# Binary Cross-Entropy Loss (for classification, NOT MSE!)
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model_lr_pt.parameters(), lr=0.1)

losses = []
for epoch in range(500):
    y_pred = model_lr_pt(X_spam_pt)       # Forward pass
    loss = criterion(y_pred, y_spam_pt)    # Cross-entropy loss
    optimizer.zero_grad()                  # Clear gradients
    loss.backward()                        # Compute gradients
    optimizer.step()                       # Update weights
    losses.append(loss.item())

# Check predictions
with torch.no_grad():
    test_input = torch.tensor([[4.0, 1.0]])
    pred = model_lr_pt(test_input)
    print(f"P(spam) for [4, 1]: {pred.item():.3f}")
    print(f"Decision: {'SPAM' if pred.item() > 0.5 else 'NOT SPAM'}")

plt.plot(losses, linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('BCE Loss')
plt.title('Logistic Regression Training (PyTorch)')
plt.grid(alpha=0.3); plt.show()

## 12. Cross-Entropy Loss

In [ ]:
# Cross-entropy: -[y*log(p) + (1-y)*log(1-p)]
# Penalizes confident wrong predictions severely

p = np.linspace(0.01, 0.99, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# When true label = 1
loss_1 = -np.log(p)
ax1.plot(p, loss_1, linewidth=3, color='#2196F3')
ax1.set_xlabel('Predicted P(spam)'); ax1.set_ylabel('Loss')
ax1.set_title('True label = SPAM (y=1)'); ax1.grid(alpha=0.3)
ax1.annotate('Confident & correct\n(low loss)', xy=(0.95, 0.1), fontsize=11, color='green')
ax1.annotate('Confident & WRONG\n(huge loss!)', xy=(0.05, 3), fontsize=11, color='red')

# When true label = 0
loss_0 = -np.log(1 - p)
ax2.plot(p, loss_0, linewidth=3, color='#E91E63')
ax2.set_xlabel('Predicted P(spam)'); ax2.set_ylabel('Loss')
ax2.set_title('True label = NOT SPAM (y=0)'); ax2.grid(alpha=0.3)
ax2.annotate('Confident & correct\n(low loss)', xy=(0.02, 0.1), fontsize=11, color='green')
ax2.annotate('Confident & WRONG\n(huge loss!)', xy=(0.7, 3), fontsize=11, color='red')

plt.suptitle('Cross-Entropy Loss', fontsize=15, y=1.02)
plt.tight_layout(); plt.show()
print("Key insight: Being confident AND wrong = very high loss!")

## 13. Polynomial Features: Beyond Straight Lines

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Non-linear data: ice cream sales vs temperature
np.random.seed(42)
temp = np.array([15, 20, 25, 30, 35, 40])
sales = np.array([10, 15, 25, 50, 90, 140]) + np.random.randn(6) * 3

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear fit
lin = LinearRegression().fit(temp.reshape(-1, 1), sales)
x_plot = np.linspace(12, 42, 100)
ax1.scatter(temp, sales, s=100, color='black', zorder=5)
ax1.plot(x_plot, lin.predict(x_plot.reshape(-1, 1)), color='#E91E63', linewidth=2)
ax1.set_title('Linear: Poor fit'); ax1.set_xlabel('Temperature'); ax1.set_ylabel('Sales')
ax1.grid(alpha=0.3)

# Polynomial fit (degree=2)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(temp.reshape(-1, 1))
poly_model = LinearRegression().fit(X_poly, sales)
X_plot_poly = poly.transform(x_plot.reshape(-1, 1))
ax2.scatter(temp, sales, s=100, color='black', zorder=5)
ax2.plot(x_plot, poly_model.predict(X_plot_poly), color='#2196F3', linewidth=2)
ax2.set_title('Polynomial (degree=2): Great fit!'); ax2.set_xlabel('Temperature'); ax2.set_ylabel('Sales')
ax2.grid(alpha=0.3)

plt.suptitle('Linear vs Polynomial Features', fontsize=15, y=1.02)
plt.tight_layout(); plt.show()
print("Polynomial features let linear models learn curves!")

## Summary

| Slide Topic | What we verified |
|---|---|
| House price linear regression | weight=0.04, price = 0.04 * size |
| Noisy data + 3 lines | SSE comparison, Line B wins |
| Residuals and MSE | Residual table, SSE = sum of squared errors |
| Normal equation | theta_hat = (X'X)^{-1}X'y matches sklearn |
| Gradient descent | Converges to same answer iteratively |
| Learning rate | Too small/right/large comparison |
| Feature scaling | StandardScaler, fit on train only |
| sklearn vs PyTorch | Same answer for linear regression |
| Sigmoid | Squashes any z to (0,1) |
| Logistic regression (sklearn) | .predict() and .predict_proba() |
| Logistic regression (PyTorch) | nn.Module class + BCELoss training |
| Cross-entropy | Confident + wrong = huge loss |
| Polynomial features | Linear models can learn curves |